In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
%pip install -q -U "transformers>=4.46.0" "peft>=0.13.0" "accelerate>=1.0.0" "bitsandbytes>=0.44.0" "safetensors>=0.4.5"

In [4]:
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:
import os
import json
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 322

def set_seed(seed: int = 322):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))

Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA capability: (7, 5)


In [ ]:
 !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
CANDIDATE_DATA_DIRS = [
    Path("./processed_spider_sft"),
    Path("/kaggle/working/processed_spider_sft"),
]

kaggle_input = Path("/kaggle/input/datasets/olllllllll/spider")
if kaggle_input.exists():
    CANDIDATE_DATA_DIRS.extend([p.parent for p in kaggle_input.glob("**/train_sft.jsonl")])

DATA_DIR = None
for candidate in CANDIDATE_DATA_DIRS:
    if (candidate / "train_sft.jsonl").exists() and (candidate / "dev_sft.jsonl").exists():
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Не найдена папка "
    )

TRAIN_PATH = DATA_DIR / "train_sft.jsonl"
DEV_PATH = DATA_DIR / "dev_sft.jsonl"
DEV_EVAL_PATH = DATA_DIR / "dev_eval_prompts.jsonl"

print("DATA_DIR:", DATA_DIR)
print("TRAIN_PATH:", TRAIN_PATH)
print("DEV_PATH:", DEV_PATH)
print("DEV_EVAL_PATH exists:", DEV_EVAL_PATH.exists())

DATA_DIR: /kaggle/input/datasets/olllllllll/spider
TRAIN_PATH: /kaggle/input/datasets/olllllllll/spider/train_sft.jsonl
DEV_PATH: /kaggle/input/datasets/olllllllll/spider/dev_sft.jsonl
DEV_EVAL_PATH exists: True


In [7]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": str(TRAIN_PATH),
        "validation": str(DEV_PATH),
    },
)

print(dataset)
print("Train columns:", dataset["train"].column_names)
print("Validation columns:", dataset["validation"].column_names)
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'split', 'db_id', 'question', 'schema', 'sql', 'text'],
        num_rows: 8659
    })
    validation: Dataset({
        features: ['id', 'split', 'db_id', 'question', 'schema', 'sql', 'text'],
        num_rows: 1034
    })
})
Train columns: ['id', 'split', 'db_id', 'question', 'schema', 'sql', 'text']
Validation columns: ['id', 'split', 'db_id', 'question', 'schema', 'sql', 'text']
Train size: 8659
Validation size: 1034


In [ ]:

import math
import random
import re
from collections import Counter

import pandas as pd
from datasets import Dataset, DatasetDict

AUGMENTATION_SEED = 322
EXTRA_SAMPLE_RATIO = 0.0

MAX_SAMPLING_WEIGHT = 2.5

random.seed(AUGMENTATION_SEED)

STRICT_INSTRUCTION = """### Instruction:
You are a Text-to-SQL assistant.
Generate exactly one valid SQLite SQL query for the given question.
Use only tables and columns that are explicitly present in the database schema.
When a JOIN is needed, use the explicit relationships listed below.
Return only the SQL query."""

def normalize_sql(sql: str) -> str:
    return " ".join(str(sql).lower().strip().split())


def count_pattern(pattern: str, text: str) -> int:
    return len(re.findall(pattern, text, flags=re.IGNORECASE))


def extract_sql_features(sql: str) -> dict:
    sql_norm = normalize_sql(sql)

    select_count = count_pattern(r"\bselect\b", sql_norm)
    join_count = count_pattern(r"\bjoin\b", sql_norm)
    set_operation_count = count_pattern(
        r"\b(intersect|union|except)\b",
        sql_norm,
    )
    aggregate_count = count_pattern(
        r"\b(count|sum|avg|min|max)\s*\(",
        sql_norm,
    )

    return {
        "join_count": join_count,
        "nested_select_count": max(0, select_count - 1),
        "set_operation_count": set_operation_count,
        "aggregate_count": aggregate_count,
        "has_group_by": bool(re.search(r"\bgroup\s+by\b", sql_norm)),
        "has_having": bool(re.search(r"\bhaving\b", sql_norm)),
        "has_distinct": bool(re.search(r"\bdistinct\b", sql_norm)),
        "has_membership_subquery": bool(
            re.search(r"\b(not\s+in|in|exists)\s*\(", sql_norm)
        ),
    }


def calculate_complexity_score(sql: str) -> int:

    features = extract_sql_features(sql)

    score = 0
    score += features["join_count"]

    if features["join_count"] >= 2:
        score += 1

    score += 2 * features["nested_select_count"]
    score += 3 * features["set_operation_count"]

    if features["has_group_by"]:
        score += 1

    if features["has_having"]:
        score += 2

    if features["has_distinct"]:
        score += 1

    if features["has_membership_subquery"]:
        score += 1

    score += min(
        max(features["aggregate_count"] - 1, 0),
        2,
    )

    return score


def score_to_difficulty_proxy(score: int) -> str:
    if score <= 1:
        return "simple"

    if score <= 3:
        return "moderate"

    if score <= 5:
        return "hard"

    return "extra"


def calculate_sampling_weight(sql: str) -> float:
    score = calculate_complexity_score(sql)
    difficulty = score_to_difficulty_proxy(score)
    features = extract_sql_features(sql)

    weight = {
        "simple": 1.0,
        "moderate": 2.0,
        "hard": 3.0,
        "extra": 4.0,
    }[difficulty]

    if features["set_operation_count"] > 0:
        weight += 1.0

    if features["nested_select_count"] > 0:
        weight += 0.5

    if features["has_having"]:
        weight += 0.5

    if features["join_count"] >= 2:
        weight += 0.5

    return min(weight, MAX_SAMPLING_WEIGHT)


def extract_explicit_relationships(schema: str) -> list[str]:
    relationships = []

    create_table_pattern = re.compile(
        r"CREATE\s+TABLE\s+([`\"\[\]\w]+)\s*\((.*?)\)\s*;",
        flags=re.IGNORECASE | re.DOTALL,
    )

    fk_pattern = re.compile(
        r"FOREIGN\s+KEY\s*\(([^)]+)\)\s*"
        r"REFERENCES\s+([`\"\[\]\w]+)\s*\(([^)]+)\)",
        flags=re.IGNORECASE,
    )

    for table_match in create_table_pattern.finditer(schema):
        source_table = table_match.group(1).strip("`\"[] ")
        table_body = table_match.group(2)

        for fk_match in fk_pattern.finditer(table_body):
            source_columns = [
                value.strip("`\"[] ")
                for value in fk_match.group(1).split(",")
            ]

            target_table = fk_match.group(2).strip("`\"[] ")

            target_columns = [
                value.strip("`\"[] ")
                for value in fk_match.group(3).split(",")
            ]

            relationships.append(
                f"{source_table}.{', '.join(source_columns)} "
                f"-> {target_table}.{', '.join(target_columns)}"
            )

    return sorted(set(relationships))


def render_relationship_section(schema: str) -> str:
    relationships = extract_explicit_relationships(schema)

    if not relationships:
        return "### Explicit relationships:\n- No foreign-key relationships are listed."

    return (
        "### Explicit relationships:\n"
        + "\n".join(
            f"- {relationship}"
            for relationship in relationships
        )
    )


def build_text2sql_prompt(
    schema: str,
    question: str,
    sql: str | None = None,
) -> str:

    prompt = f"""{STRICT_INSTRUCTION}

### Database schema:
{schema.strip()}

{render_relationship_section(schema)}

### Question:
{question.strip()}

### SQL:
"""

    if sql is None:
        return prompt

    return prompt + str(sql).strip()


def prepare_example(
    row: dict,
    *,
    split_name: str,
) -> dict:
    row = dict(row)

    complexity_score = calculate_complexity_score(row["sql"])
    difficulty_proxy = score_to_difficulty_proxy(complexity_score)
    sampling_weight = calculate_sampling_weight(row["sql"])

    row["text"] = build_text2sql_prompt(
        schema=row["schema"],
        question=row["question"],
        sql=row["sql"],
    )

    row["complexity_score"] = complexity_score
    row["difficulty_proxy"] = difficulty_proxy
    row["sampling_weight"] = sampling_weight
    row["dataset_split"] = split_name

    return row


def augment_train_rows(
    rows: list[dict],
    *,
    extra_sample_ratio: float = EXTRA_SAMPLE_RATIO,
    seed: int = AUGMENTATION_SEED,
) -> list[dict]:

    rng = random.Random(seed)

    sql_group_sizes = Counter(
        normalize_sql(row["sql"])
        for row in rows
    )

    prepared_rows = []

    for original_index, row in enumerate(rows):
        prepared = prepare_example(
            row,
            split_name="train",
        )

        normalized_sql = normalize_sql(
            prepared["sql"]
        )

        group_size = sql_group_sizes[
            normalized_sql
        ]

        adjusted_weight = (
            prepared["sampling_weight"]
            / math.sqrt(group_size)
        )

        prepared["sql_group_size"] = group_size
        prepared["adjusted_sampling_weight"] = adjusted_weight
        prepared["augmentation_kind"] = "original"
        prepared["source_row_index"] = original_index

        prepared_rows.append(prepared)

    extra_count = int(
        len(prepared_rows)
        * extra_sample_ratio
    )

    sampled_rows = rng.choices(
        population=prepared_rows,
        weights=[
            row["adjusted_sampling_weight"]
            for row in prepared_rows
        ],
        k=extra_count,
    )

    augmented_rows = list(prepared_rows)

    for replica_index, sampled_row in enumerate(sampled_rows):
        replica = dict(sampled_row)
        replica["augmentation_kind"] = "weighted_replica"
        replica["augmentation_replica_index"] = replica_index
        augmented_rows.append(replica)

    rng.shuffle(augmented_rows)

    return augmented_rows


def prepare_validation_rows(
    rows: list[dict],
) -> list[dict]:

    prepared_rows = []

    for original_index, row in enumerate(rows):
        prepared = prepare_example(
            row,
            split_name="validation",
        )

        normalized_sql = normalize_sql(
            prepared["sql"]
        )

        prepared["sql_group_size"] = 1
        prepared["adjusted_sampling_weight"] = prepared["sampling_weight"]
        prepared["augmentation_kind"] = "validation_original"
        prepared["source_row_index"] = original_index
        prepared["augmentation_replica_index"] = -1

        prepared_rows.append(prepared)

    return prepared_rows


def build_inference_prompt(example: dict) -> str:

    return build_text2sql_prompt(
        schema=example["schema"],
        question=example["question"],
        sql=None,
    )


raw_train_rows = [
    dict(row)
    for row in dataset["train"]
]

raw_validation_rows = [
    dict(row)
    for row in dataset["validation"]
]

augmented_train_rows = augment_train_rows(
    raw_train_rows,
)

prepared_validation_rows = prepare_validation_rows(
    raw_validation_rows,
)

for row in augmented_train_rows:
    row.setdefault(
        "augmentation_replica_index",
        -1,
    )

dataset = DatasetDict(
    {
        "train": Dataset.from_list(
            augmented_train_rows
        ),
        "validation": Dataset.from_list(
            prepared_validation_rows
        ),
    }
)

print("Original train size:", len(raw_train_rows))
print("Augmented train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))
print()

train_audit = pd.DataFrame(
    augmented_train_rows
)

print("Difficulty distribution in augmented train:")
display(
    train_audit[
        "difficulty_proxy"
    ]
    .value_counts()
    .rename_axis("difficulty_proxy")
    .reset_index(name="count")
)

print("Augmentation distribution:")
display(
    train_audit[
        "augmentation_kind"
    ]
    .value_counts()
    .rename_axis("augmentation_kind")
    .reset_index(name="count")
)

print("Most complex examples:")
display(
    train_audit[
        [
            "db_id",
            "question",
            "sql",
            "complexity_score",
            "difficulty_proxy",
            "sampling_weight",
            "sql_group_size",
            "adjusted_sampling_weight",
            "augmentation_kind",
        ]
    ]
    .sort_values(
        by=[
            "complexity_score",
            "adjusted_sampling_weight",
        ],
        ascending=False,
    )
    .head(30)
)


Original train size: 8659
Augmented train size: 8659
Validation size: 1034

Difficulty distribution in augmented train:


,difficulty_proxy,count
0,simple,4656
1,moderate,2196
2,hard,1188
3,extra,619


Augmentation distribution:


,augmentation_kind,count
0,original,8659


Most complex examples:


,db_id,question,sql,complexity_score,difficulty_proxy,sampling_weight,sql_group_size,adjusted_sampling_weight,augmentation_kind
27,geo,what is the largest state that borders the sta...,SELECT state_name FROM state WHERE area = ( SE...,17,extra,2.5,1,2.500000,original
769,insurance_policies,Which claims caused more than 2 settlements or...,"SELECT T1.Date_Claim_Made , T1.Claim_id FROM C...",14,extra,2.5,2,1.767767,original
1974,college_1,Find the first name of student who is taking c...,SELECT T1.stu_fname FROM student AS T1 JOIN en...,14,extra,2.5,2,1.767767,original
7147,insurance_policies,Find the claims that led to more than two sett...,"SELECT T1.Date_Claim_Made , T1.Claim_id FROM C...",14,extra,2.5,2,1.767767,original
7737,college_1,What are the first names of all students takin...,SELECT T1.stu_fname FROM student AS T1 JOIN en...,14,extra,2.5,2,1.767767,original
2233,geo,what is the smallest city in the largest state,SELECT city_name FROM city WHERE population = ...,13,extra,2.5,1,2.500000,original
2459,geo,what is the longest river in the largest state,SELECT river_name FROM river WHERE LENGTH = ( ...,13,extra,2.5,1,2.500000,original
3048,geo,what is the longest river in the smallest stat...,SELECT river_name FROM river WHERE LENGTH = ( ...,13,extra,2.5,1,2.500000,original
5037,geo,what is the population of the largest city in ...,SELECT population FROM city WHERE population =...,13,extra,2.5,1,2.500000,original
7194,geo,what is the smallest state through which the l...,SELECT state_name FROM state WHERE area = ( SE...,13,extra,2.5,1,2.500000,original


In [ ]:
sample = dataset["train"][0]

print("Keys:", sample.keys())
print("=" * 80)
print(sample["text"][:3000])

Keys: dict_keys(['id', 'split', 'db_id', 'question', 'schema', 'sql', 'text', 'complexity_score', 'difficulty_proxy', 'sampling_weight', 'dataset_split', 'sql_group_size', 'adjusted_sampling_weight', 'augmentation_kind', 'source_row_index', 'augmentation_replica_index'])
### Instruction:
You are a Text-to-SQL assistant.
Generate exactly one valid SQLite SQL query for the given question.
Use only tables and columns that are explicitly present in the database schema.
When a JOIN is needed, use the explicit relationships listed below.
Return only the SQL query.

### Database schema:
CREATE TABLE product (
    product_id NUMBER,
    product TEXT,
    dimensions TEXT,
    dpi NUMBER,
    pages_per_minute_color NUMBER,
    max_page_size TEXT,
    interface TEXT,
    PRIMARY KEY (product_id)
);

CREATE TABLE store (
    Store_ID NUMBER,
    Store_Name TEXT,
    Type TEXT,
    Area_size NUMBER,
    Number_of_product_category NUMBER,
    Ranking NUMBER,
    PRIMARY KEY (Store_ID)
);

CREATE TAB

In [10]:
OUTPUT_DIR = Path("./qwen25-coder-15b-spider-qlora-r16")
ADAPTER_DIR = OUTPUT_DIR / "adapter"
LOG_DIR = OUTPUT_DIR / "logs"
HISTORY_PATH = OUTPUT_DIR / "training_history.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
from transformers import AutoTokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("pad_token:", tokenizer.pad_token)
print("eos_token:", tokenizer.eos_token)
print("model_max_length:", tokenizer.model_max_length)

==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

pad_token: <|PAD_TOKEN|>
eos_token: <|im_end|>
model_max_length: 32768


In [ ]:
!pip install --upgrade bitsandbytes

In [ ]:
from peft import LoraConfig, get_peft_model

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=322,
)


Unsloth 2026.6.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
!pip install --upgrade trl


In [ ]:
!pip install --upgrade transformers

In [ ]:
from transformers import DataCollatorForSeq2Seq

MAX_SEQ_LENGTH = 2048
SQL_MARKER = "### SQL:\n"

def split_prompt_and_answer(text):
    if SQL_MARKER not in text:
        raise ValueError("SQL marker not found in text")

    prompt_part, answer_part = text.split(SQL_MARKER, 1)

    prompt = prompt_part + SQL_MARKER
    answer = answer_part.strip()

    return prompt, answer


def tokenize_with_sql_labels(example):
    prompt, answer = split_prompt_and_answer(example["text"])

    answer = answer + tokenizer.eos_token

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
    )["input_ids"]

    answer_ids = tokenizer(
        answer,
        add_special_tokens=False,
    )["input_ids"]

    input_ids = prompt_ids + answer_ids
    attention_mask = [1] * len(input_ids)

    labels = [-100] * len(prompt_ids) + answer_ids.copy()

    if len(input_ids) > MAX_SEQ_LENGTH:
        input_ids = input_ids[:MAX_SEQ_LENGTH]
        attention_mask = attention_mask[:MAX_SEQ_LENGTH]
        labels = labels[:MAX_SEQ_LENGTH]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "length": len(input_ids),
        "answer_tokens": sum(x != -100 for x in labels),
    }


tokenized_dataset = dataset.map(
    tokenize_with_sql_labels,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing with SQL-only labels",
)

print(tokenized_dataset)

Tokenizing with SQL-only labels:   0%|          | 0/8659 [00:00<?, ? examples/s]

Tokenizing with SQL-only labels:   0%|          | 0/1034 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'length', 'answer_tokens'],
        num_rows: 8659
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'length', 'answer_tokens'],
        num_rows: 1034
    })
})


In [14]:
import numpy as np
import pandas as pd

def check_split(split):
    lengths = np.array(tokenized_dataset[split]["length"])
    answer_tokens = np.array(tokenized_dataset[split]["answer_tokens"])

    return {
        "split": split,
        "count": len(lengths),
        "mean_length": lengths.mean(),
        "p95_length": np.percentile(lengths, 95),
        "max_length": lengths.max(),
        "mean_answer_tokens": answer_tokens.mean(),
        "zero_answer_tokens": int((answer_tokens == 0).sum()),
    }

pd.DataFrame([
    check_split("train"),
    check_split("validation"),
])

,split,count,mean_length,p95_length,max_length,mean_answer_tokens,zero_answer_tokens
0,train,8659,512.870077,1160.0,2048,33.603649,82
1,validation,1034,384.039652,869.7,952,28.911992,0


In [15]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=None,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,

    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=100,

    fp16=True,
    bf16=False,
    

    optim="paged_adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=0.3,

    report_to="none",
    dataloader_num_workers=4,           
    dataloader_pin_memory=True,          
    seed=322,
    data_seed=322,
)

In [4]:
import transformers

In [17]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Trainer is ready")

Trainer is ready


In [18]:
train_result = trainer.train()

print("Training finished")
print(train_result)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,659 | Num Epochs = 1 | Total steps = 542
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,0.221486,0.266905
100,0.171768,0.259515
150,0.137356,0.246314
200,0.146522,0.246335
250,0.131916,0.244042
300,0.124287,0.247733
350,0.103476,0.248352
400,0.130473,0.248170
450,0.103560,0.247749
500,0.100288,0.244859


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Training finished
TrainOutput(global_step=542, training_loss=0.1451884629101771, metrics={'train_runtime': 7829.3847, 'train_samples_per_second': 1.106, 'train_steps_per_second': 0.069, 'total_flos': 5.819394979434394e+16, 'train_loss': 0.1451884629101771, 'epoch': 1.0})


In [ ]:
!nvidia-smi

In [ ]:
final_eval_metrics = trainer.evaluate()
print(final_eval_metrics)

In [ ]:
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

trainer.state.save_to_json(str(OUTPUT_DIR / "trainer_state.json"))

history = pd.DataFrame(trainer.state.log_history)
history.to_csv(HISTORY_PATH, index=False)

with open(OUTPUT_DIR / "training_history.json", "w", encoding="utf-8") as f:
    json.dump(trainer.state.log_history, f, ensure_ascii=False, indent=2)

metrics = {
    "initial_eval": initial_eval_metrics,
    "final_eval": final_eval_metrics,
    "train_result": train_result.metrics,
    "model_name": MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "seed": SEED,
    "data_dir": str(DATA_DIR),
}

with open(OUTPUT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Saved adapter to:", ADAPTER_DIR)
print("Saved history to:", HISTORY_PATH)
print("Saved metrics to:", OUTPUT_DIR / "metrics.json")

In [ ]:
history = pd.read_csv(HISTORY_PATH)

display(history.tail(20))

loss_cols = [col for col in ["loss", "eval_loss", "learning_rate", "epoch", "step"] if col in history.columns]
display(history[loss_cols].dropna(how="all").tail(50))

In [ ]:

import matplotlib.pyplot as plt

if "loss" in history.columns:
    train_history = history.dropna(subset=["loss"])
    if len(train_history) > 0:
        plt.figure(figsize=(8, 5))
        plt.plot(train_history["step"], train_history["loss"])
        plt.xlabel("Step")
        plt.ylabel("Train loss")
        plt.title("Training loss")
        plt.grid(True)
        plt.savefig(OUTPUT_DIR / "train_loss.png", dpi=150, bbox_inches="tight")
        plt.show()

if "eval_loss" in history.columns:
    eval_history = history.dropna(subset=["eval_loss"])
    if len(eval_history) > 0:
        plt.figure(figsize=(8, 5))
        plt.plot(eval_history["step"], eval_history["eval_loss"])
        plt.xlabel("Step")
        plt.ylabel("Eval loss")
        plt.title("Validation loss")
        plt.grid(True)
        plt.savefig(OUTPUT_DIR / "eval_loss.png", dpi=150, bbox_inches="tight")
        plt.show()

In [ ]:

from peft import PeftModel

def clean_generated_sql(full_output: str) -> str:
    if "### SQL:" in full_output:
        sql = full_output.split("### SQL:")[-1]
    else:
        sql = full_output

    sql = sql.strip()
    sql = sql.replace("```sql", "").replace("```", "").strip()

    for marker in ["###", "\nQuestion:", "\nDatabase schema:"]:
        if marker in sql:
            sql = sql.split(marker)[0].strip()

    return sql.strip()

def generate_sql(prompt_or_example, max_new_tokens: int = 256) -> str:

    FastLanguageModel.for_inference(model)
    model.eval()

    if isinstance(prompt_or_example, dict):
        prompt = build_inference_prompt(prompt_or_example)
    else:
        prompt = prompt_or_example

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return clean_generated_sql(decoded)

if DEV_EVAL_PATH.exists():
    with open(DEV_EVAL_PATH, "r", encoding="utf-8") as f:
        eval_example = json.loads(next(f))

    pred_sql = generate_sql(eval_example)

    print("QUESTION:")
    print(eval_example["question"])
    print("\nPRED SQL:")
    print(pred_sql)
    print("\nGOLD SQL:")
    print(eval_example["sql"])
else:
    print("dev_eval_prompts.jsonl не найден, пропускаем генерацию.")

In [ ]:

N_DEV_PREDICTIONS = 50
PRED_DIR = OUTPUT_DIR / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)
PRED_PATH = PRED_DIR / f"dev_predictions_first_{N_DEV_PREDICTIONS}.jsonl"

if DEV_EVAL_PATH.exists():
    predictions = []
    with open(DEV_EVAL_PATH, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= N_DEV_PREDICTIONS:
                break

            item = json.loads(line)
            pred_sql = generate_sql(item["text"])

            predictions.append({
                "id": item.get("id"),
                "db_id": item.get("db_id"),
                "question": item.get("question"),
                "gold_sql": item.get("sql"),
                "pred_sql": pred_sql,
            })

    with open(PRED_PATH, "w", encoding="utf-8") as f:
        for item in predictions:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print("Saved predictions to:", PRED_PATH)
    display(pd.DataFrame(predictions).head())
else:
    print("dev_eval_prompts.jsonl не найден, predictions не сохранены.")

In [ ]:
ARCHIVE_BASE = shutil.make_archive(
    base_name=str(OUTPUT_DIR),
    format="zip",
    root_dir=str(OUTPUT_DIR),
)

print("Archive saved:", ARCHIVE_BASE)

In [19]:
from pathlib import Path

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"

ADAPTER_DIR = Path("/kaggle/input/models/replekxt/lora-7b-150/pytorch/default/1")
EVAL_PROMPTS_FILE = Path("/kaggle/input/datasets/olllllllll/spider/dev_eval_prompts.jsonl")

print("Adapter exists:", ADAPTER_DIR.exists())
print("Eval file exists:", EVAL_PROMPTS_FILE.exists())
print(list(ADAPTER_DIR.iterdir())[:10])

Adapter exists: True
Eval file exists: True
[PosixPath('/kaggle/input/models/replekxt/lora-7b-150/pytorch/default/1/adapter_model.safetensors'), PosixPath('/kaggle/input/models/replekxt/lora-7b-150/pytorch/default/1/adapter_config.json'), PosixPath('/kaggle/input/models/replekxt/lora-7b-150/pytorch/default/1/tokenizer.json'), PosixPath('/kaggle/input/models/replekxt/lora-7b-150/pytorch/default/1/tokenizer_config.json')]


In [20]:
import torch
from unsloth import FastLanguageModel
from peft import PeftModel

MAX_SEQ_LENGTH = 2048

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)

FastLanguageModel.for_inference(model)

print("Model with adapter loaded")

==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Model with adapter loaded


In [21]:
import json

examples = []

with open(EVAL_PROMPTS_FILE, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 10:
            break
        examples.append(json.loads(line))

print("Loaded examples:", len(examples))
print(examples[0].keys())

Loaded examples: 10
dict_keys(['id', 'split', 'db_id', 'question', 'schema', 'sql', 'text'])


In [22]:
def clean_generated_sql(text: str) -> str:
    text = text.strip()

    if "```sql" in text:
        text = text.split("```sql", 1)[-1]
        text = text.split("```", 1)[0]
    elif "```" in text:
        text = text.split("```", 1)[-1]
        text = text.split("```", 1)[0]

    stop_markers = [
        "### Explanation:",
        "Explanation:",
        "\n\nExplanation",
        "\n###",
        "\nNote:",
        "\nThe query",
        "\nThis query",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker, 1)[0]

    text = text.strip()

    if ";" in text:
        text = text.split(";", 1)[0].strip()

    return text

In [ ]:
def generate_sql(example, max_new_tokens=128):
    prompt = build_inference_prompt(example)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw_sql = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    clean_sql = clean_generated_sql(raw_sql)

    return raw_sql, clean_sql

In [24]:
for i, ex in enumerate(examples):
    raw_pred, clean_pred = generate_sql(ex)

    print("=" * 120)
    print("EXAMPLE:", i)
    print("DB:", ex["db_id"])
    print("QUESTION:", ex["question"])

    print("\nRAW PRED:")
    print(raw_pred)

    print("\nCLEAN PRED:")
    print(clean_pred)

    print("\nGOLD:")
    print(ex["sql"])

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

EXAMPLE: 0
DB: concert_singer
QUESTION: How many singers do we have?

RAW PRED:
SELECT count(*) FROM singer

CLEAN PRED:
SELECT count(*) FROM singer

GOLD:
SELECT count(*) FROM singer


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EXAMPLE: 1
DB: concert_singer
QUESTION: What is the total number of singers?

RAW PRED:
SELECT count(*) FROM singer

CLEAN PRED:
SELECT count(*) FROM singer

GOLD:
SELECT count(*) FROM singer


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

EXAMPLE: 6
DB: concert_singer
QUESTION: Show the name and the release year of the song by the youngest singer.

RAW PRED:
SELECT T1.Name , T1.Song_release_year FROM singer AS T1 JOIN (SELECT MIN(Age) FROM singer) AS T2 ON T1.Age = T2.MIN

CLEAN PRED:
SELECT T1.Name , T1.Song_release_year FROM singer AS T1 JOIN (SELECT MIN(Age) FROM singer) AS T2 ON T1.Age = T2.MIN

GOLD:
SELECT song_name , song_release_year FROM singer ORDER BY age LIMIT 1


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EXAMPLE: 7
DB: concert_singer
QUESTION: What are the names and release years for all the songs of the youngest singer?

RAW PRED:
SELECT T1.song_name , T1.song_release_year FROM singer AS T2 JOIN song AS T1 ON T2.singer_id = T1.singer_id WHERE T2.age = (SELECT MIN(age) FROM singer)

CLEAN PRED:
SELECT T1.song_name , T1.song_release_year FROM singer AS T2 JOIN song AS T1 ON T2.singer_id = T1.singer_id WHERE T2.age = (SELECT MIN(age) FROM singer)

GOLD:
SELECT song_name , song_release_year FROM singer ORDER BY age LIMIT 1


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EXAMPLE: 8
DB: concert_singer
QUESTION: What are all distinct countries where singers above age 20 are from?

RAW PRED:
SELECT DISTINCT country FROM singer WHERE age > 20

CLEAN PRED:
SELECT DISTINCT country FROM singer WHERE age > 20

GOLD:
SELECT DISTINCT country FROM singer WHERE age > 20
EXAMPLE: 9
DB: concert_singer
QUESTION: What are  the different countries with singers above age 20?

RAW PRED:
SELECT DISTINCT country FROM singer WHERE age > 20

CLEAN PRED:
SELECT DISTINCT country FROM singer WHERE age > 20

GOLD:
SELECT DISTINCT country FROM singer WHERE age > 20


In [25]:
def normalize_sql_for_string_match(sql: str) -> str:
    sql = sql.strip().lower()
    sql = sql.replace(";", "")
    sql = " ".join(sql.split())
    return sql

N = 50
eval_examples = []

with open(EVAL_PROMPTS_FILE, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= N:
            break
        eval_examples.append(json.loads(line))

results = []

for ex in eval_examples:
    raw_pred, clean_pred = generate_sql(ex)

    pred_norm = normalize_sql_for_string_match(clean_pred)
    gold_norm = normalize_sql_for_string_match(ex["sql"])

    results.append({
        "id": ex["id"],
        "db_id": ex["db_id"],
        "question": ex["question"],
        "gold_sql": ex["sql"],
        "raw_pred_sql": raw_pred,
        "pred_sql": clean_pred,
        "string_exact_match": pred_norm == gold_norm,
    })

string_em = sum(r["string_exact_match"] for r in results) / len(results)

print("N:", len(results))
print("Simple string exact match:", round(string_em, 4))

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

N: 50
Simple string exact match: 0.46


In [38]:
!git clone https://github.com/taoyds/spider.git /kaggle/working/spider_official

fatal: destination path '/kaggle/working/spider_official' already exists and is not an empty directory.


In [27]:
import json
from pathlib import Path
from tqdm.auto import tqdm

EVAL_PROMPTS_FILE = Path("/kaggle/input/datasets/olllllllll/spider/dev_eval_prompts.jsonl")

PREDICTIONS_DIR = Path("/kaggle/working/spider_predictions")
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_JSONL = PREDICTIONS_DIR / "dev_predictions.jsonl"

In [28]:
def clean_generated_sql(text: str) -> str:
    text = text.strip()

    if "```sql" in text:
        text = text.split("```sql", 1)[-1]
        text = text.split("```", 1)[0]
    elif "```" in text:
        text = text.split("```", 1)[-1]
        text = text.split("```", 1)[0]

    stop_markers = [
        "### Explanation:",
        "Explanation:",
        "\n\nExplanation",
        "\n###",
        "\nNote:",
        "\nThe query",
        "\nThis query",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker, 1)[0]

    text = text.strip()

    if ";" in text:
        text = text.split(";", 1)[0].strip()

    return " ".join(text.split())

In [ ]:
import torch

def generate_sql(example, max_new_tokens=128):
    prompt = build_inference_prompt(example)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw_sql = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    clean_sql = clean_generated_sql(raw_sql)

    return raw_sql, clean_sql


In [ ]:
import json
import torch
from tqdm.auto import tqdm
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)
model.eval()

examples = []

with open(EVAL_PROMPTS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        examples.append(json.loads(line))

print("Dev examples:", len(examples))

with open(PREDICTIONS_JSONL, "w", encoding="utf-8") as f:
    for ex in tqdm(examples):
        raw_pred, clean_pred = generate_sql(ex)

        item = {
            "id": ex["id"],
            "db_id": ex["db_id"],
            "question": ex["question"],
            "gold_sql": ex["sql"],
            "raw_pred_sql": raw_pred,
            "pred_sql": clean_pred,
        }

        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved:", PREDICTIONS_JSONL)

Dev examples: 1034


  0%|          | 0/1034 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Saved: /kaggle/working/spider_predictions/dev_predictions.jsonl


In [31]:
EVAL_DIR = Path("/kaggle/working/spider_eval_files")
EVAL_DIR.mkdir(parents=True, exist_ok=True)

GOLD_SQL_PATH = EVAL_DIR / "gold.sql"
PRED_SQL_PATH = EVAL_DIR / "pred.sql"

def clean_for_eval(sql: str) -> str:
    sql = sql.strip()
    sql = sql.replace("\n", " ")
    sql = " ".join(sql.split())
    if sql.endswith(";"):
        sql = sql[:-1].strip()
    return sql

with open(PREDICTIONS_JSONL, "r", encoding="utf-8") as source, \
     open(GOLD_SQL_PATH, "w", encoding="utf-8") as gold_file, \
     open(PRED_SQL_PATH, "w", encoding="utf-8") as pred_file:

    for line in source:
        item = json.loads(line)

        gold_sql = clean_for_eval(item["gold_sql"])
        pred_sql = clean_for_eval(item["pred_sql"])
        db_id = item["db_id"]

        gold_file.write(f"{gold_sql}\t{db_id}\n")
        pred_file.write(f"{pred_sql}\n")

print("Gold:", GOLD_SQL_PATH)
print("Pred:", PRED_SQL_PATH)

Gold: /kaggle/working/spider_eval_files/gold.sql
Pred: /kaggle/working/spider_eval_files/pred.sql


In [32]:
!head -n 3 /kaggle/working/spider_eval_files/gold.sql
!head -n 3 /kaggle/working/spider_eval_files/pred.sql

SELECT count(*) FROM singer	concert_singer
SELECT count(*) FROM singer	concert_singer
SELECT name , country , age FROM singer ORDER BY age DESC	concert_singer
SELECT count(*) FROM singer
SELECT count(*) FROM singer
SELECT name , country , age FROM singer ORDER BY age DESC


In [33]:
SPIDER_DATA_DIR = Path("/kaggle/input/datasets/olllllllll/spider-orig/spider")
SPIDER_DB_DIR = SPIDER_DATA_DIR / "database"
SPIDER_TABLES = SPIDER_DATA_DIR / "tables.json"

print("DB dir exists:", SPIDER_DB_DIR.exists(), SPIDER_DB_DIR)
print("Tables exists:", SPIDER_TABLES.exists(), SPIDER_TABLES)

DB dir exists: True /kaggle/input/datasets/olllllllll/spider-orig/spider/database
Tables exists: True /kaggle/input/datasets/olllllllll/spider-orig/spider/tables.json


In [40]:
!python /kaggle/working/spider_official/evaluation.py \
  --gold /kaggle/working/spider_eval_files/gold.sql \
  --pred /kaggle/working/spider_eval_files/pred.sql \
  --db {SPIDER_DB_DIR} \
  --table {SPIDER_TABLES} \
  --etype all

eval_err_num:1
medium pred: SELECT T1.Name , T1.Song_release_year FROM singer AS T1 JOIN (SELECT MIN(Age) FROM singer) AS T2 ON T1.Age = T2.MIN
medium gold: SELECT song_name , song_release_year FROM singer ORDER BY age LIMIT 1

eval_err_num:2
medium pred: SELECT T1.song_name , T1.song_release_year FROM singer AS T2 JOIN song AS T1 ON T2.singer_id = T1.singer_id WHERE T2.age = (SELECT MIN(age) FROM singer)
medium gold: SELECT song_name , song_release_year FROM singer ORDER BY age LIMIT 1

medium pred: SELECT max(capacity) , avg(capacity) FROM stadium
medium gold: select max(capacity), average from stadium

medium pred: SELECT name , capacity FROM stadium WHERE average = ( SELECT MAX ( average ) FROM stadium )
medium gold: SELECT name , capacity FROM stadium ORDER BY average DESC LIMIT 1

medium pred: SELECT name , capacity FROM stadium WHERE average = ( SELECT MAX ( average ) FROM stadium )
medium gold: SELECT name , capacity FROM stadium ORDER BY average DESC LIMIT 1

medium pred: SELE

In [35]:
!git clone https://github.com/taoyds/test-suite-sql-eval.git /kaggle/working/test_suite_eval

Cloning into '/kaggle/working/test_suite_eval'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 61 (delta 20), reused 16 (delta 16), pack-reused 31 (from 2)
Receiving objects: 100% (61/61), 619.62 KiB | 5.16 MiB/s, done.
Resolving deltas: 100% (24/24), done.


In [36]:
!python /kaggle/working/test_suite_eval/evaluation.py \
  --gold /kaggle/working/spider_eval_files/gold.sql \
  --pred /kaggle/working/spider_eval_files/pred.sql \
  --db {SPIDER_DB_DIR} \
  --table {SPIDER_TABLES} \
  --etype all

/kaggle/working/test_suite_eval/exec_eval.py:127: SyntaxWarning: invalid escape sequence '\s'
  "YEAR\s*\(\s*CURDATE\s*\(\s*\)\s*\)\s*", "2020", query, flags=re.IGNORECASE
/kaggle/working/test_suite_eval/parse.py:57: SyntaxWarning: invalid escape sequence '\d'
  float_nums = re.findall("[-+]?\d*\.\d+", query)
/kaggle/working/test_suite_eval/parse.py:62: SyntaxWarning: invalid escape sequence '\d'
  int_nums = [i.strip() for i in re.findall("[^tT]\d+", query)]
/kaggle/working/test_suite_eval/parse.py:70: SyntaxWarning: invalid escape sequence '\d'
  table = re.findall("[Tt]\d+\.", tok)
/kaggle/working/test_suite_eval/parse.py:206: SyntaxWarning: invalid escape sequence '\.'
  for table, col, val1, val2 in re.findall('(?:([^\.\s]*)\.)?([^\.\s]+) between ([^\s;]+) and ([^\s;]+)', query, re.IGNORECASE):
medium pred: SELECT T1.Name , T1.Song_release_year FROM singer AS T1 JOIN (SELECT MIN(Age) FROM singer) AS T2 ON T1.Age = T2.MIN
medium gold: SELECT song_name , song_release_year FROM singe